# MASR Fine-tuning on ViMD (Preprocessed Fixed Data)

**Model:** PhoWhisper + TS-RoPE + HyperSD = MASR
**Data:** `vimd_processed_fixed.zip` (da fix audio > 30s)
**HOP_LENGTH:** 320 (khop Whisper encoder output 1500 frames/30s)


In [ ]:
# Cell 1: Setup
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

!pip install -q transformers accelerate soundfile librosa jiwer scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.6 MB/s eta 0:00:0000:0100:01


In [ ]:
# Cell 2: Imports
import os, json, glob, random, zipfile, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


2026-02-18 11:27:23.638984: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771414044.023846      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771414044.134061      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771414045.017660      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771414045.017696      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771414045.017699      55 computation_placer.cc:177] computation placer alr

Device: cuda


In [ ]:
# Cell 3: Config
class Config:
    # Model
    BASE_MODEL = 'vinai/PhoWhisper-small'
    MAX_SPEAKERS = 4

    # Data - dung file FIXED (da cat audio > 30s)
    DATASET_ZIP = '/content/drive/MyDrive/vimd_processed_fixed.zip'
    EXTRACT_DIR = '/kaggle/input/datasets/cnghuy/mars-data/vimd_processed_fixed'

    # Audio
    SAMPLE_RATE = 16000
    HOP_LENGTH = 320   # Whisper encoder: 30s -> 1500 frames. 30*16000/320 = 1500. KHOP!
    MAX_AUDIO_LENGTH = 30.0  # Whisper limit

    # Training
    NUM_EPOCHS = 15
    BATCH_SIZE = 4
    LR = 1e-6
    WEIGHT_DECAY = 0.01
    GRAD_ACCUM = 2
    FP16 = True

    # Loss weights
    ASR_WEIGHT = 1.0
    SD_WEIGHT = 0.3

    # Pretrained (optional)
    PRETRAINED_PATH = '/kaggle/input/models/cnghuy/marscheckpoint5/pytorch/default/1/ckpt_ep5.pt'

    # Output
    OUTPUT_DIR = '/kaggle/working/masr_vimd_results'

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print(f"Config:")
print(f"  Model: {cfg.BASE_MODEL}")
print(f"  Data: {cfg.DATASET_ZIP}")
print(f"  HOP_LENGTH: {cfg.HOP_LENGTH} (-> {int(cfg.MAX_AUDIO_LENGTH * cfg.SAMPLE_RATE / cfg.HOP_LENGTH)} frames/30s)")
print(f"  Epochs: {cfg.NUM_EPOCHS}, Batch: {cfg.BATCH_SIZE}x{cfg.GRAD_ACCUM}")
print(f"  LR: {cfg.LR}, FP16: {cfg.FP16}")


Config:
  Model: vinai/PhoWhisper-small
  Data: /content/drive/MyDrive/vimd_processed_fixed.zip
  HOP_LENGTH: 320 (-> 1500 frames/30s)
  Epochs: 15, Batch: 4x2
  LR: 1e-06, FP16: True


---
## 1. Load Preprocessed Data


In [ ]:
# Cell 4: Extract data
with open(f"{cfg.EXTRACT_DIR}/metadata.json", "r", encoding="utf-8") as f:
    all_metadata = json.load(f)

print(f"Loaded {len(all_metadata)} samples")

# Quick validation
n_check = min(50, len(all_metadata))
n_ok = 0
for item in all_metadata[:n_check]:
    apath = os.path.join(cfg.EXTRACT_DIR, item.get('audio_file', ''))
    if os.path.exists(apath):
        a, sr = sf.read(apath)
        dur = len(a) / sr
        act_len = len(item.get('activity', []))
        exp_frames = int(len(a) / cfg.HOP_LENGTH)
        if dur <= 30.5 and abs(act_len - exp_frames) <= 10:
            n_ok += 1

print(f"\nData validation ({n_check} samples): {n_ok}/{n_check} OK")
if n_ok < n_check * 0.9:
    print("WARNING: Data quality issues! Run check_vimd_data.ipynb first.")
else:
    print("Data quality: GOOD")


Loaded 15023 samples

Data validation (50 samples): 38/50 OK


In [ ]:
# Cell 5: PreprocessedViMDDataset
class PreprocessedViMDDataset(Dataset):
    def __init__(self, metadata_list, base_dir, max_audio_length=30.0, sr=16000):
        self.metadata = metadata_list
        self.base_dir = base_dir
        self.max_samples = int(max_audio_length * sr)
        self.sr = sr

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        item = self.metadata[idx]

        # Load audio
        audio_path = os.path.join(self.base_dir, item["audio_file"])
        audio, sr = sf.read(audio_path)
        audio = audio.astype(np.float32)

        # Truncate to 30s max
        if len(audio) > self.max_samples:
            audio = audio[:self.max_samples]
        elif len(audio) < self.max_samples:
            audio = np.pad(audio, (0, self.max_samples - len(audio)))

        # HOP=320: 30s*16000/320 = 1500 frames = Whisper encoder output
        n_frames = int(len(audio) / cfg.HOP_LENGTH)

        activity = np.array(item["activity"], dtype=np.float32)
        class_labels = np.array(item["class_labels"], dtype=np.int64)

        # Align activity/class_labels to n_frames
        if activity.shape[0] < n_frames:
            activity = np.pad(activity, ((0, n_frames - activity.shape[0]), (0, 0)))
            class_labels = np.pad(class_labels, (0, n_frames - class_labels.shape[0]))
        elif activity.shape[0] > n_frames:
            activity = activity[:n_frames]
            class_labels = class_labels[:n_frames]

        return {
            'audio': audio,
            'transcript': item.get('transcript', '') or '.',
            'n_frames': n_frames,
            'activity': activity,
            'class_labels': class_labels,
            'has_overlap': 1 if item.get('has_overlap', False) else 0
        }

# Split
train_meta, val_meta = train_test_split(all_metadata, test_size=0.1, random_state=42)
train_ds = PreprocessedViMDDataset(train_meta, cfg.EXTRACT_DIR, cfg.MAX_AUDIO_LENGTH)
val_ds = PreprocessedViMDDataset(val_meta, cfg.EXTRACT_DIR, cfg.MAX_AUDIO_LENGTH)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

# Test
s = train_ds[0]
print(f"  audio: {s['audio'].shape}")
print(f"  activity: {s['activity'].shape}")
print(f"  class_labels: {s['class_labels'].shape}")
print(f"  n_frames: {s['n_frames']}")
print(f"  transcript: {s['transcript'][:80]}...")


Train: 13520, Val: 1503
  audio: (480000,)
  activity: (1500, 4)
  class_labels: (1500,)
  n_frames: 1500
  transcript: <|spk1|> Tôi thì rất là đam mê về những nghệ thuật, nhất là những cái đồ vật dụn...


---
## 2. Model


In [ ]:
# Cell 6: MASR Model
class TSROPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.d_model = d_model
        self.div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
    def forward(self, x, speaker_shift=0):
        seq_len = x.size(1)
        t = torch.arange(seq_len, device=x.device).float().unsqueeze(1)
        t_shifted = t + speaker_shift
        angles = t_shifted * self.div_term
        sin = torch.sin(angles)
        cos = torch.cos(angles)
        x_left = x[..., 0::2]
        x_right = x[..., 1::2]
        x_rotated = torch.zeros_like(x)
        x_rotated[..., 0::2] = x_left * cos - x_right * sin
        x_rotated[..., 1::2] = x_left * sin + x_right * cos
        return x_rotated
class TSPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)
    def forward(self, x):
        return x + self.pe[:x.size(1)]

class HyperSDModule(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_speakers=4, num_classes=5):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(0.1)
        )
        self.speaker_head = nn.Linear(hidden_dim, num_speakers)
        self.class_head = nn.Linear(hidden_dim, num_classes)
    def forward(self, h):
        h = self.proj(h)
        return self.speaker_head(h), self.class_head(h)

class MASR(nn.Module):
    def __init__(self, phowhisper, hidden_dim=256, num_speakers=4, asr_w=1.0, sd_w=0.3):
        super().__init__()
        self.phowhisper = phowhisper
        self.encoder_dim = phowhisper.config.d_model
        self.ts_pos = TSPositionalEncoding(self.encoder_dim)
        self.hyper_sd = HyperSDModule(self.encoder_dim, hidden_dim, num_speakers)
        self.asr_w = asr_w; self.sd_w = sd_w

    def forward(self, input_features, labels=None, speaker_activity_gt=None, sd_class_labels=None):
        enc = self.phowhisper.model.encoder(input_features)
        h = self.ts_pos(enc.last_hidden_state)
        spk_logits, cls_logits = self.hyper_sd(h)

        asr_loss = None
        if labels is not None:
            asr_loss = self.phowhisper(input_features=input_features, labels=labels).loss

        sd_loss = None
        if speaker_activity_gt is not None:
            ml = min(spk_logits.size(1), speaker_activity_gt.size(1))
            sd_loss = F.binary_cross_entropy_with_logits(spk_logits[:,:ml], speaker_activity_gt[:,:ml])

        cls_loss = None
        if sd_class_labels is not None:
            ml = min(cls_logits.size(1), sd_class_labels.size(1))
            cls_loss = F.cross_entropy(cls_logits[:,:ml].reshape(-1, cls_logits.size(-1)),
                                       sd_class_labels[:,:ml].reshape(-1))

        total = None
        if asr_loss is not None:
            total = self.asr_w * asr_loss
            if sd_loss is not None: total += self.sd_w * sd_loss
            if cls_loss is not None: total += self.sd_w * 0.5 * cls_loss

        return {'loss': total, 'asr_loss': asr_loss, 'sd_loss': sd_loss, 'cls_loss': cls_loss,
                'speaker_logits': spk_logits, 'class_logits': cls_logits}

    def generate(self, input_features, **kw):
        return self.phowhisper.generate(input_features, **kw)

print("Model classes defined")


Model classes defined


In [ ]:
# Cell 7: Load PhoWhisper + Build MASR
print(f"Loading {cfg.BASE_MODEL}...")
processor = WhisperProcessor.from_pretrained(cfg.BASE_MODEL)
phowhisper = WhisperForConditionalGeneration.from_pretrained(cfg.BASE_MODEL)

# Add speaker tokens
special_tokens = [f'<|spk{i}|>' for i in range(1, cfg.MAX_SPEAKERS + 1)]
processor.tokenizer.add_tokens(special_tokens)
phowhisper.resize_token_embeddings(len(processor.tokenizer))
print(f"Added tokens: {special_tokens}")

masr = MASR(phowhisper, 256, cfg.MAX_SPEAKERS, cfg.ASR_WEIGHT, cfg.SD_WEIGHT).to(device)
total_params = sum(p.numel() for p in masr.parameters()) / 1e6
trainable = sum(p.numel() for p in masr.parameters() if p.requires_grad) / 1e6
print(f"MASR: {total_params:.1f}M params ({trainable:.1f}M trainable)")


Loading vinai/PhoWhisper-small...


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Added tokens: ['<|spk1|>', '<|spk2|>', '<|spk3|>', '<|spk4|>']
MASR: 240.8M params (239.6M trainable)


### (Optional) Load Pretrained Weights

Neu da train `train_masr_single_phase.ipynb` -> chay cell duoi.
**Bo qua neu muon train tu dau.**


In [ ]:
# Cell 8: [OPTIONAL] Load pretrained weights
PRETRAINED_PATH = "/kaggle/input/models/cnghuy/marscheckpoint5/pytorch/default/1/ckpt_ep5.pt"

if PRETRAINED_PATH and os.path.exists(PRETRAINED_PATH):
    print(f'Loading pretrained: {PRETRAINED_PATH}...')
    ckpt = torch.load(PRETRAINED_PATH, map_location=device, weights_only=False)
    state_dict = ckpt.get('model', ckpt.get('model_state_dict', ckpt))

    filtered_state_dict = {}
    for k, v in state_dict.items():
        if k in masr.state_dict() and v.shape == masr.state_dict()[k].shape:
            filtered_state_dict[k] = v
        else:
            print(f'Skipped: {k} | ckpt shape: {v.shape} vs model shape: {masr.state_dict().get(k, "NOT FOUND")}')

    missing, unexpected = masr.load_state_dict(filtered_state_dict, strict=False)
    print(f'Loaded! Missing: {len(missing)}, Unexpected: {len(unexpected)}')
else:
    print(f'No pretrained at {PRETRAINED_PATH}')
    print('Training from base PhoWhisper')

Loading pretrained: /kaggle/input/models/cnghuy/marscheckpoint5/pytorch/default/1/ckpt_ep5.pt...
Loaded! Missing: 0, Unexpected: 0


---
## 3. DataLoader


In [ ]:
# Cell 9: Collator + DataLoader
class MASRCollator:
    def __init__(self, processor, max_speakers=4):
        self.proc = processor
        self.vocab_size = len(processor.tokenizer)

    def __call__(self, batch):
        audios = [b['audio'] for b in batch]
        feats = self.proc.feature_extractor(audios, sampling_rate=16000, return_tensors='pt').input_features

        mf = max(b['n_frames'] for b in batch)
        act = np.stack([np.pad(b['activity'], ((0, mf-b['n_frames']),(0,0)))
                        if b['n_frames'] < mf else b['activity'][:mf] for b in batch])
        cls = np.stack([np.pad(b['class_labels'], (0, mf-b['n_frames']))
                        if b['n_frames'] < mf else b['class_labels'][:mf] for b in batch])

        txts = [b['transcript'] for b in batch]
        labels = self.proc.tokenizer(txts, return_tensors='pt', padding=True,
                                     truncation=True, max_length=448).input_ids
        labels = labels.clamp(0, self.vocab_size - 1)
        labels = labels.masked_fill(labels == self.proc.tokenizer.pad_token_id, -100)

        return {
            'input_features': feats, 'labels': labels,
            'speaker_activity_gt': torch.from_numpy(act).float(),
            'sd_class_labels': torch.from_numpy(cls).long(),
            'transcripts': txts,
            'has_overlap': torch.tensor([b['has_overlap'] for b in batch])
        }

collator = MASRCollator(processor, cfg.MAX_SPEAKERS)
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          collate_fn=collator, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        collate_fn=collator, num_workers=2, pin_memory=True)

print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches")

# Test batch
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  input_features: {batch['input_features'].shape}")
print(f"  labels: {batch['labels'].shape}")
print(f"  speaker_activity_gt: {batch['speaker_activity_gt'].shape}")
print(f"  sd_class_labels: {batch['sd_class_labels'].shape}")


Train: 3380 batches, Val: 376 batches

Batch shapes:
  input_features: torch.Size([4, 80, 3000])
  labels: torch.Size([4, 331])
  speaker_activity_gt: torch.Size([4, 1500, 4])
  sd_class_labels: torch.Size([4, 1500])


---
## 4. Evaluation Metrics


In [ ]:
# Cell 10: Evaluation functions
from jiwer import wer as compute_wer, cer as compute_cer

def compute_der_components(pred_activity, gt_activity, threshold=0.5):
    pred_binary = (pred_activity > threshold).astype(float)
    ml = min(len(pred_binary), len(gt_activity))
    pred_binary = pred_binary[:ml]
    gt = gt_activity[:ml]

    pred_any = pred_binary.max(axis=-1) if pred_binary.ndim > 1 else pred_binary
    gt_any = gt.max(axis=-1) if gt.ndim > 1 else gt

    total = max(gt_any.sum(), 1)
    miss = ((gt_any == 1) & (pred_any == 0)).sum() / total
    fa = ((gt_any == 0) & (pred_any == 1)).sum() / max(gt_any.shape[0], 1)

    confusion = 0
    if pred_binary.ndim > 1 and gt.ndim > 1:
        both_active = (gt_any == 1) & (pred_any == 1)
        n_both = max(both_active.sum(), 1)
        spk_match = (pred_binary[both_active] == gt[both_active]).all(axis=-1) if both_active.any() else np.array([True])
        confusion = 1 - spk_match.mean()

    der = miss + fa + confusion

    # Precision, Recall, F1
    tp = ((gt_any == 1) & (pred_any == 1)).sum()
    fp = ((gt_any == 0) & (pred_any == 1)).sum()
    fn = ((gt_any == 1) & (pred_any == 0)).sum()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    return {'DER': der, 'E_miss': miss, 'E_fa': fa, 'E_conf': confusion,
            'Precision': precision, 'Recall': recall, 'F1': f1}

@torch.no_grad()
def evaluate_full(masr, val_loader, processor, device, max_batches=50):
    masr.eval()
    total_loss, total_asr, total_sd = 0, 0, 0
    all_preds, all_refs, all_der = [], [], []
    n = 0

    for i, batch in enumerate(tqdm(val_loader, desc="Evaluating")):
        if i >= max_batches: break
        out = masr(input_features=batch['input_features'].to(device),
                   labels=batch['labels'].to(device),
                   speaker_activity_gt=batch['speaker_activity_gt'].to(device),
                   sd_class_labels=batch['sd_class_labels'].to(device))
        if out['loss']: total_loss += out['loss'].item()
        if out['asr_loss']: total_asr += out['asr_loss'].item()
        if out['sd_loss']: total_sd += out['sd_loss'].item()
        n += 1

        # ASR
        gen_ids = masr.generate(batch['input_features'].to(device), max_new_tokens=256)
        preds = processor.batch_decode(gen_ids, skip_special_tokens=True)
        refs = batch['transcripts']
        clean = lambda t: t.replace('<|spk1|>','').replace('<|spk2|>','').replace('<|spk3|>','').replace('<|spk4|>','').strip()
        all_preds.extend([clean(p) for p in preds])
        all_refs.extend([clean(r) for r in refs])

        # DER
        spk_probs = torch.sigmoid(out['speaker_logits']).cpu().numpy()
        gt_act = batch['speaker_activity_gt'].numpy()
        for b in range(spk_probs.shape[0]):
            all_der.append(compute_der_components(spk_probs[b], gt_act[b]))

    # Filter empty
    pairs = [(p, r) for p, r in zip(all_preds, all_refs) if p.strip() and r.strip()]
    if pairs:
        ps, rs = zip(*pairs)
        wer_val = compute_wer(list(rs), list(ps))
        cer_val = compute_cer(list(rs), list(ps))
    else:
        wer_val, cer_val = 1.0, 1.0

    avg_der = {}
    if all_der:
        for k in all_der[0]:
            avg_der[k] = np.mean([d[k] for d in all_der])

    masr.train()
    return {
        'WER': wer_val, 'CER': cer_val,
        'loss': total_loss/max(n,1),
        'asr_loss': total_asr/max(n,1),
        'sd_loss': total_sd/max(n,1),
        **{f'SD_{k}': v for k, v in avg_der.items()}
    }

print("Evaluation functions defined")


Evaluation functions defined


---
## 5. Training (voi Resume tu Checkpoint)


In [ ]:
# Cell 11: Training Loop
from torch.optim.lr_scheduler import OneCycleLR

optimizer = torch.optim.AdamW(masr.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
total_steps = len(train_loader) * cfg.NUM_EPOCHS // cfg.GRAD_ACCUM
scheduler = OneCycleLR(optimizer, max_lr=cfg.LR, total_steps=max(total_steps, 1), pct_start=0.1)
scaler = GradScaler('cuda') if cfg.FP16 else None

history = {'total':[], 'asr':[], 'sd':[], 'val':[], 'wer':[], 'cer':[], 'der':[],
           'precision':[], 'recall':[], 'f1':[], 'e_miss':[], 'e_fa':[], 'e_conf':[]}
best_wer = float('inf')
start_epoch = 0

# ======== RESUME TU CHECKPOINT ========
ckpt_files = sorted(glob.glob(f'{cfg.OUTPUT_DIR}/ckpt_ep*.pt'))
if ckpt_files:
    latest_ckpt = ckpt_files[-1]
    print(f'Found {len(ckpt_files)} checkpoint(s)')
    print(f'Loading: {os.path.basename(latest_ckpt)}...')
    ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
    masr.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    if 'scheduler' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler'])
    history = ckpt.get('history', history)
    start_epoch = ckpt['epoch'] + 1
    best_wer = ckpt.get('best_wer', float('inf'))
    if best_wer == float('inf') and history.get('wer'):
        best_wer = min(history['wer'])
    if 'scheduler' not in ckpt:
        steps_done = start_epoch * len(train_loader) // cfg.GRAD_ACCUM
        for _ in range(min(steps_done, total_steps - 1)):
            scheduler.step()
    print(f'Resuming from epoch {start_epoch + 1}/{cfg.NUM_EPOCHS}')
    print(f'Best WER: {best_wer*100:.1f}%')
else:
    print('No checkpoint found -> Training from scratch')

print("=" * 60)
print(f"TRAINING MASR on ViMD (fixed data)")
print(f"Config: HOP_LENGTH={cfg.HOP_LENGTH}, Epochs: {start_epoch+1}->{cfg.NUM_EPOCHS}")
print(f"Batch: {cfg.BATCH_SIZE}x{cfg.GRAD_ACCUM}, LR: {cfg.LR}")
print("=" * 60)

for epoch in range(start_epoch, cfg.NUM_EPOCHS):
    masr.train()
    t_loss, t_asr, t_sd = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.NUM_EPOCHS}")

    for step, batch in enumerate(pbar):
        if cfg.FP16:
            with autocast('cuda'):
                out = masr(input_features=batch['input_features'].to(device),
                           labels=batch['labels'].to(device),
                           speaker_activity_gt=batch['speaker_activity_gt'].to(device),
                           sd_class_labels=batch['sd_class_labels'].to(device))
                loss = out['loss'] / cfg.GRAD_ACCUM
            scaler.scale(loss).backward()
        else:
            out = masr(input_features=batch['input_features'].to(device),
                       labels=batch['labels'].to(device),
                       speaker_activity_gt=batch['speaker_activity_gt'].to(device),
                       sd_class_labels=batch['sd_class_labels'].to(device))
            loss = out['loss'] / cfg.GRAD_ACCUM
            loss.backward()

        t_loss += loss.item() * cfg.GRAD_ACCUM
        if out['asr_loss']: t_asr += out['asr_loss'].item()
        if out['sd_loss']: t_sd += out['sd_loss'].item()

        if (step + 1) % cfg.GRAD_ACCUM == 0:
            if cfg.FP16:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(masr.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(masr.parameters(), 1.0)
                optimizer.step()
            scheduler.step(); optimizer.zero_grad()

        pbar.set_postfix({'loss': f'{loss.item()*cfg.GRAD_ACCUM:.3f}',
                          'asr': f'{out["asr_loss"].item():.2f}' if out['asr_loss'] else '-',
                          'sd': f'{out["sd_loss"].item():.3f}' if out['sd_loss'] else '-'})

    # Evaluate
    print(f"\nEvaluating epoch {epoch+1}...")
    m = evaluate_full(masr, val_loader, processor, device, max_batches=30)

    nb = len(train_loader)
    history['total'].append(t_loss/nb); history['asr'].append(t_asr/nb); history['sd'].append(t_sd/nb)
    history['val'].append(m['loss']); history['wer'].append(m['WER']); history['cer'].append(m['CER'])
    history['der'].append(m.get('SD_DER',0))
    history['precision'].append(m.get('SD_Precision',0)); history['recall'].append(m.get('SD_Recall',0))
    history['f1'].append(m.get('SD_F1',0))
    history['e_miss'].append(m.get('SD_E_miss',0)); history['e_fa'].append(m.get('SD_E_fa',0))
    history['e_conf'].append(m.get('SD_E_conf',0))

    print(f"Epoch {epoch+1}: Loss={t_loss/nb:.4f}")
    print(f"  WER={m['WER']*100:.1f}% | CER={m['CER']*100:.1f}% | DER={m.get('SD_DER',0)*100:.1f}%")
    print(f"  P={m.get('SD_Precision',0)*100:.1f}% R={m.get('SD_Recall',0)*100:.1f}% F1={m.get('SD_F1',0)*100:.1f}%")

    if m['WER'] < best_wer:
        best_wer = m['WER']
        torch.save(masr.state_dict(), f"{cfg.OUTPUT_DIR}/masr_best.pt")
        print(f"  New best! WER={best_wer*100:.1f}%")

    # Save checkpoint
    torch.save({'epoch': epoch, 'model': masr.state_dict(), 'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(), 'history': history,
                'best_wer': best_wer}, f"{cfg.OUTPUT_DIR}/ckpt_ep{epoch+1}.pt")
    print(f"  Checkpoint saved: ckpt_ep{epoch+1}.pt")

print(f"\nDone! Best WER: {best_wer*100:.2f}%")


No checkpoint found -> Training from scratch
TRAINING MASR on ViMD (fixed data)
Config: HOP_LENGTH=320, Epochs: 1->15
Batch: 4x2, LR: 1e-06


Epoch 1/15: 100%|██████████| 3380/3380 [58:23<00:00,  1.04s/it, loss=0.495, asr=0.46, sd=0.057]



Evaluating epoch 1...


Evaluating:   0%|          | 0/376 [00:00<?, ?it/s]Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Evaluating:   8%|▊         | 30/376 [04:23<50:35,  8.77s/it] 


Epoch 1: Loss=0.4842
  WER=68.8% | CER=57.4% | DER=4.0%
  P=99.5% R=99.7% F1=99.6%
  New best! WER=68.8%
  Checkpoint saved: ckpt_ep1.pt


...


Evaluating epoch 6...


Evaluating:   8%|▊         | 30/376 [04:24<50:45,  8.80s/it]


Epoch 6: Loss=0.3470
  WER=68.7% | CER=57.8% | DER=1.5%
  P=99.9% R=99.7% F1=99.8%
  Checkpoint saved: ckpt_ep6.pt


Epoch 7/15:  17%|█▋        | 578/3380 [10:03<48:43,  1.04s/it, loss=0.434, asr=0.42, sd=0.028] 


KeyboardInterrupt: 

---
## 6. Visualization


In [ ]:
# Cell 12: Training curves
if len(history['wer']) > 0:
    epochs_range = range(1, len(history['wer']) + 1)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Loss
    ax = axes[0, 0]
    ax.plot(epochs_range, history['total'], 'b-o', label='Train Total')
    ax.plot(epochs_range, history['asr'], 'r--', label='Train ASR')
    ax.plot(epochs_range, history['sd'], 'g--', label='Train SD')
    ax.plot(epochs_range, history['val'], 'm-s', label='Val Loss')
    ax.set_title('Loss'); ax.legend(); ax.grid(True, alpha=0.3)

    # WER & CER
    ax = axes[0, 1]
    ax.plot(epochs_range, [w*100 for w in history['wer']], 'r-o', label='WER')
    ax.plot(epochs_range, [c*100 for c in history['cer']], 'b-s', label='CER')
    ax.set_title('WER & CER (%)'); ax.legend(); ax.grid(True, alpha=0.3)

    # DER
    ax = axes[0, 2]
    ax.plot(epochs_range, [d*100 for d in history['der']], 'g-o', label='DER')
    ax.set_title('DER (%)'); ax.legend(); ax.grid(True, alpha=0.3)

    # P/R/F1
    ax = axes[1, 0]
    ax.plot(epochs_range, [p*100 for p in history['precision']], 'b-o', label='Precision')
    ax.plot(epochs_range, [r*100 for r in history['recall']], 'r-s', label='Recall')
    ax.plot(epochs_range, [f*100 for f in history['f1']], 'g-^', label='F1')
    ax.set_title('P / R / F1 (%)'); ax.legend(); ax.grid(True, alpha=0.3)

    # DER breakdown
    ax = axes[1, 1]
    ax.plot(epochs_range, [m*100 for m in history['e_miss']], 'r-o', label='E_miss')
    ax.plot(epochs_range, [f*100 for f in history['e_fa']], 'b-s', label='E_fa')
    ax.plot(epochs_range, [c*100 for c in history['e_conf']], 'g-^', label='E_conf')
    ax.set_title('DER Breakdown (%)'); ax.legend(); ax.grid(True, alpha=0.3)

    # Summary table
    ax = axes[1, 2]
    ax.axis('off')
    best_ep = np.argmin(history['wer']) + 1
    summary = f"Best Epoch: {best_ep}\n"
    summary += f"WER: {min(history['wer'])*100:.1f}%\n"
    summary += f"CER: {history['cer'][best_ep-1]*100:.1f}%\n"
    summary += f"DER: {history['der'][best_ep-1]*100:.1f}%\n"
    summary += f"F1:  {history['f1'][best_ep-1]*100:.1f}%"
    ax.text(0.5, 0.5, summary, transform=ax.transAxes, fontsize=14,
            verticalalignment='center', horizontalalignment='center',
            fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightblue'))

    plt.tight_layout()
    plt.savefig(f"{cfg.OUTPUT_DIR}/training_curves.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No training history to plot")


In [ ]:
# Cell 13: Sample predictions
masr.eval()
masr.load_state_dict(torch.load(f"{cfg.OUTPUT_DIR}/masr_best.pt", map_location=device, weights_only=False))

batch = next(iter(val_loader))
with torch.no_grad():
    gen_ids = masr.generate(batch['input_features'].to(device), max_new_tokens=256)
    preds = processor.batch_decode(gen_ids, skip_special_tokens=True)

print("Sample Predictions vs Ground Truth:")
print("=" * 60)
for i in range(min(5, len(preds))):
    print(f"\n[{i+1}] REF: {batch['transcripts'][i][:100]}")
    print(f"    PRED: {preds[i][:100]}")


In [ ]:
import json
import os
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

metadata = {
    "title": "MASR Model Checkpoints",
    "id": "cnghuy/masr-model-checkpoints",
    "licenses": [{"name": "CC0-1.0"}]
}

metadata_path = f"{cfg.OUTPUT_DIR}/dataset-metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f)

try:
    api.dataset_create_version(
        folder=cfg.OUTPUT_DIR,
        version_notes="Auto update best weight and checkpoints",
        dir_mode="tar"
    )
except Exception:
    api.dataset_create_new(
        folder=cfg.OUTPUT_DIR,
        dir_mode="tar"
    )